# Text clustering on the IMDB dataset

In [18]:
# Importing necessary packages
import pandas as pd
import re
from tqdm import tqdm
import pickle

import nltk
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger') 

from sentence_transformers import SentenceTransformer

import gensim


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\danie\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\danie\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\danie\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [45]:
df = pd.read_csv("movie_review.csv")[["id", "sentiment", "review"]]
df

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\""The Classic War of the Worlds\"" by Timothy H..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...
...,...,...,...
4995,3720_2,0,Is this film a joke? Is it a comedy? Surely it...
4996,4229_10,1,Don't waste time reading my review. Go out and...
4997,8042_3,0,"The secret is...this movie blows. Sorry, but i..."
4998,9669_9,1,After reading the original play I thought it w...


# Preprocessing
Preprocessing is performed on each review. The following steps are taken:
- Lowercasing
- HTML tags are removed
- Punctuation is removed (without single quotes, as some stopwords contain single quotes, such as "you'll")
- Stopwords are removed
- Single quotes are removed
- Lemmatization using POS tagging (bringing words back to their root)

In [46]:
# Defining utilities that do not have to be defined all the time
le = WordNetLemmatizer()

# Define a function to get correct position input for the Lemmatizer.
def get_pos_code(x):
    if x.startswith('J'):
        return 'a'
    elif x.startswith('V'):
        return 'v'
    elif x.startswith('N'):
        return 'n'
    elif x.startswith('R'):
        return 'r'
    else:
        return 'n'

def preprocess_document(doc: str):
    """
    This function preprocesses a textual document into a bag of words.
    Applied steps:
    - Lowercasing
    - Removing HTML tags
    - Removing stopwords
    - Removing punctuation
    - Lemmatization

    """
    # Converting to lowercase
    doc = doc.lower()

    # Removing HTML tags
    doc = re.sub("</?[^>]+>", "", doc)

    # Removing punctuation, but ' (single quotes) are not removed (as stopwords may contain them.)
    punct = str.maketrans("", "", '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~')
    doc = doc.translate(punct)

    # Removing stopwords
    doc = doc.split(" ")
    doc = [word for word in doc if word not in stopwords.words("English")]

    # Removing the single quote
    doc = " ".join(doc)
    punct = str.maketrans("", "", "'")
    doc = doc.translate(punct)
    doc = word_tokenize(doc)

    # Lemmatization
    doc = pos_tag(doc) # Pos tagging to improve lemmatization process.
    doc = [le.lemmatize(word, get_pos_code(tag)) for word, tag in doc]

    # Joining again into single string for final output
    doc = " ".join(doc)

    return doc
    
    


In [ ]:
df["review"] = tqdm(df["review"].apply(preprocess_document))
# This takes a long time. 

100%|██████████| 5000/5000 [00:00<?, ?it/s]


In [ ]:
# Saving the preprocessed reviews to use them later (without havign to do the whole preprocessing again)
df.to_csv("movie_review_preprocessed.csv")

In [8]:
# Here load the preprocessed DataFrame (so you don't have to perform it each time)
df = pd.read_csv("movie_review_preprocessed.csv", index_col = 0)
df

,id,sentiment,review
0,5814_8,1,stuff go moment mj ive start listen music watc...
1,2381_9,1,classic war world timothy hines entertain film...
2,7759_3,0,film start manager nicholas bell give welcome ...
3,3630_4,0,must assume praised film great film opera ever...
4,9495_8,1,superbly trashy wondrously unpretentious 80 ex...
...,...,...,...
4995,3720_2,0,film joke comedy surely serious thriller sugge...
4996,4229_10,1,waste time read review go see astonishingly go...
4997,8042_3,0,secret isthis movie blow sorry spoilerin bad r...
4998,9669_9,1,read original play think would much difficult ...


# Text Embedding
Now, the sentences are embedded using the sentence_transformers library.

We will use the lightweight sentence-transformers/all-MiniLM-L6-v2 embedding model to generate 384-dimensional dense vector embeddings. The numerical output makes it possible to cluster using distance metrics.

In [ ]:
# Using sentence-transformers/all-MiniLM-L6-v2
model_id = "sentence-transformers/all-MiniLM-L6-v2"

# HuggingFace token is secretly stored
with open("token.txt", "r") as token_file:
    hf_token = token_file.readline()

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", token = hf_token)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2452.93it/s]


In [73]:
embeddings = model.encode(list(df["review"]))
print(f"Embedded into {len(embeddings[0])}-dimensional space")

Embedded into 384-dimensional space


In [74]:
# Pickling the embeddings (to quickly open them in new sessions)
pickle.dump(embeddings, open('embeddings.pkl', 'wb'))

In [3]:
# Unpicklings the embeddings (loading them from a file)
loaded = pickle.load(open('embeddings.pkl', 'rb'))
print(f"Loaded embeddings \n{len(loaded)} vectors, {len(loaded[0])} dimensions")

Loaded embeddings 
5000 vectors, 384 dimensions


# Topic Modeling and Text Clustering
Now, the embedded text can be clustered. The following algorithms will be used:
- Topic Modeling with Latent Dirichlet Allocation (LDA)
- GMM?

In [19]:
# Creating a corpus for Latent Dirichlet Allocation

reviews_list = [review.split() for review in list(df["review"])]
id2word = gensim.corpora.Dictionary(reviews_list)
corpus = [id2word.doc2bow(doc) for doc in reviews_list]

In [23]:
# Performing LDA
num_topics = 10

lda = gensim.models.LdaMulticore(
    corpus = corpus,
    id2word = id2word,
    num_topics = num_topics
)
lda.print_topics()

[(0,
  '0.015*"film" + 0.007*"movie" + 0.007*"one" + 0.007*"see" + 0.006*"get" + 0.006*"good" + 0.006*"make" + 0.005*"like" + 0.005*"watch" + 0.005*"time"'),
 (1,
  '0.018*"film" + 0.017*"movie" + 0.009*"one" + 0.007*"like" + 0.006*"make" + 0.005*"get" + 0.005*"good" + 0.005*"watch" + 0.004*"scene" + 0.004*"see"'),
 (2,
  '0.010*"movie" + 0.009*"like" + 0.008*"film" + 0.007*"one" + 0.005*"show" + 0.005*"good" + 0.004*"make" + 0.004*"character" + 0.004*"go" + 0.004*"time"'),
 (3,
  '0.016*"film" + 0.011*"movie" + 0.011*"one" + 0.007*"like" + 0.006*"see" + 0.006*"time" + 0.005*"get" + 0.005*"make" + 0.005*"bad" + 0.005*"well"'),
 (4,
  '0.013*"film" + 0.011*"movie" + 0.007*"good" + 0.007*"get" + 0.007*"make" + 0.006*"one" + 0.006*"see" + 0.006*"even" + 0.005*"time" + 0.005*"go"'),
 (5,
  '0.016*"film" + 0.013*"movie" + 0.008*"one" + 0.008*"make" + 0.008*"like" + 0.007*"see" + 0.006*"time" + 0.005*"good" + 0.005*"go" + 0.005*"get"'),
 (6,
  '0.010*"movie" + 0.008*"one" + 0.007*"film" + 0.

In [26]:
import pyLDAvis.gensim
import pyLDAvis

pyLDAvis.enable_notebook()

LDAvis_prepared = pyLDAvis.gensim.prepare(lda, corpus, id2word)
with open("results/ldavis_10_docs", 'wb') as f:
    pickle.dump(LDAvis_prepared, f)

In [28]:
with open("results/ldavis_10_docs", 'rb') as f:
    LDAvis_prepared = pickle.load(f)

pyLDAvis.save_html(LDAvis_prepared, './results/ldavis_10.html')
LDAvis_prepared

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
7     -0.014506  0.003381       1        1  18.804277
9     -0.010882  0.000829       2        1  17.833611
3     -0.001931  0.004445       3        1  12.694528
1     -0.005991  0.000100       4        1  10.489381
4      0.000750  0.005545       5        1  10.092851
8     -0.005126 -0.020847       6        1   7.468881
5     -0.001186  0.005378       7        1   7.301623
6      0.009899 -0.004211       8        1   6.394045
0      0.010765  0.012581       9        1   5.022654
2      0.018209 -0.007202      10        1   3.898148, topic_info=       Term         Freq        Total Category  logprob  loglift
94    movie  9369.000000  9369.000000  Default  30.0000  30.0000
50     film  9196.000000  9196.000000  Default  29.0000  29.0000
78     like  4085.000000  4085.000000  Default  28.0000  28.0000
102     one  4832.000000  4832.000000  Default  27.0000  27.0000
770    show  1918.000000  1918.000000  Default  26.0000  26.0000
..      ...          ...          ...      ...      ...      ...
53      get    76.137673  3285.572560  Topic10  -5.7439  -0.5201
117  really    64.808741  2287.610074  Topic10  -5.9050  -0.3192
125     see    77.478693  3799.169060  Topic10  -5.7265  -0.6479
52     find    53.082608  1504.614248  Topic10  -6.1046  -0.0998
139   think    55.907880  2213.167698  Topic10  -6.0528  -0.4338

[831 rows x 6 columns], token_table=       Topic      Freq      Term
term                            
43393      6  0.586651      0510
19700      7  0.611112       109
12330      4  0.460999      1895
19334     10  0.486607     1star
3002       1  0.158777         2
...      ...       ...       ...
210       10  0.039729      year
19599      3  0.526449  yetwasnt
26201      5  0.563537  youenjoy
31559      3  0.642744     zelda
48762      5  0.755537   zucovic

[1862 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[8, 10, 4, 2, 5, 9, 6, 7, 1, 3])